# Loading a Pretrained Model for Inference

This notebook demonstrates how to load a pretrained model in Hyrax and run inference without any training.
This is useful when you have a model weights file from a previous training run (or a shared model) and
want to generate latent representations or predictions for a new dataset.

The key difference from the [Getting Started](getting_started.ipynb) notebook is that we skip the
training step entirely and instead point Hyrax directly at an existing weights file.

## Install Hyrax

Before we begin we'll need to install hyrax.
You can skip this step if you're running locally and have already installed Hyrax in your virtual environment.

In [ ]:
%pip install hyrax

## Create a Hyrax instance

We start by creating a ``Hyrax`` instance, which is the main driver for all Hyrax operations.

In [ ]:
from hyrax import Hyrax

h = Hyrax()

## Specify the model

Tell Hyrax which model class to use. This must match the architecture of the pretrained weights file.
Here we use the built-in ``HyraxCNN`` model as an example.

In [ ]:
h.set_config("model.name", "HyraxCNN")

## Define the inference dataset

Configure the dataset to use for inference.
Hyrax needs an ``infer`` data group to know which data to run the model on.
Here we use the CIFAR10 test split as our inference dataset.

In [ ]:
data_request = {
    "infer": {
        "data": {
            "dataset_class": "HyraxCifarDataset",
            "data_location": "./data",
            "fields": ["image", "object_id"],
            "primary_id_field": "object_id",
            "dataset_config": {
                "use_training_data": False,
            },
        },
    },
}

h.set_config("data_request", data_request)

## Point to the pretrained weights file

Set `infer.model_weights_file` to the path of your pretrained weights file.
Hyrax saves model weights as a `.pth` file at the end of each training run inside a
timestamped results directory (e.g. `./results/20260101-120000-train-xxxx/example_model.pth`).

Replace the path below with the actual path to your weights file.
The default weights filename is ``example_model.pth``; the exact filename may vary
depending on the model and training configuration.

In [ ]:
h.set_config("infer.model_weights_file", "./results/20260101-120000-train-xxxx/example_model.pth")

## Run inference

With the model, data, and weights configured we can now call ``infer``.
Hyrax will load the pretrained weights and run a forward pass over the entire inference dataset.
The results are saved to a new timestamped directory under `./results/` and the
``ResultDataset`` object returned here can be used to access those results directly.

In [ ]:
inference_results = h.infer()

## Inspect the results

The inference results can be accessed via the returned ``ResultDataset``.
Each element in the dataset corresponds to the model output (e.g. class logits or latent vector)
for a single sample from the inference dataset.

In [ ]:
import numpy as np

# Print the shape of the output for the first sample
first_result = inference_results[0]
print(f"Number of samples: {len(inference_results)}")
print(f"Output shape for first sample: {first_result.shape}")
print(f"Output for first sample: {first_result}")

## Next steps

Once inference is complete you can:

- **Visualize** the results with ``h.umap()`` and ``h.visualize()`` — see the [Visualize](visualize.ipynb) notebook.
- **Search** for similar objects with ``h.search()`` — see the [Training to Similarity Search](hsc_train_to_similarity_search.ipynb) notebook.
- **Analyse** the output in Python directly via the ``inference_results`` object — see the [Working with Results Data](working_with_results_data.ipynb) notebook.